# 深度全连接神经网络：模块化实现 🧠

## 学习目标 🎯

完成本实验后，你将能够：

- 📦 实现深度神经网络的**参数初始化**（2 层 & L 层）
- ➡️ 构建**前向传播**模块（线性变换 + 激活函数）
- 📉 计算**交叉熵损失函数**
- ⬅️ 构建**反向传播**模块（链式法则）
- 🔄 实现**梯度下降**参数更新

> 💡 这些模块将成为你搭建任何深度网络的"乐高积木"——一旦实现，就可以反复复用！

## 目录

- [1 - 导入包](#1)
- [2 - 概览](#2)
- [3 - 参数初始化](#3)
    - [3.1 - 两层网络初始化](#3-1)
    - [3.2 - L 层网络初始化](#3-2)
- [4 - 前向传播模块](#4)
    - [4.1 - 线性前向](#4-1)
    - [4.2 - 线性激活前向](#4-2)
    - [4.3 - L 层模型前向](#4-3)
- [5 - 损失函数](#5)
- [6 - 反向传播模块](#6)
    - [6.1 - 线性反向](#6-1)
    - [6.2 - 线性激活反向](#6-2)
    - [6.3 - L 层模型反向](#6-3)
    - [6.4 - 参数更新](#6-4)
- [7 - 总结](#7)

<a name='1'></a>
## 1 - 导入包 📦

运行下方代码块，加载本实验所需的全部依赖。

- **numpy**：数值计算核心库
- **matplotlib**：可视化
- **dnn_utils**：本实验自带的激活函数工具包（sigmoid、relu 等）
- **testCases / public_tests**：自动评测模块

In [ ]:
### v1.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from testCases import *
from dnn_utils import sigmoid, sigmoid_backward, relu, relu_backward
from public_tests import *
import copy

%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0)
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

<a name='2'></a>
## 2 - 概览 🗺️

本实验将**逐模块**搭建一个通用的 L 层深度神经网络，网络结构如下：

$$[\text{LINEAR} \to \text{ReLU}]^{\times(L-1)} \to \text{LINEAR} \to \text{Sigmoid}$$

前 $L-1$ 层使用 **ReLU** 激活，最后一层使用 **Sigmoid** 输出概率（适用于二分类）。

每个模块都有**前向**和**反向**两部分：

| 模块 | 前向 | 反向 |
|------|------|------|
| 参数初始化 | ✅ | — |
| 线性变换 | ✅ | ✅ |
| 激活函数 | ✅ | ✅ |
| 损失函数 | ✅ | （内置于反向） |
| 参数更新 | — | ✅ |

> 所有中间结果（称为 **cache**）在前向传播中保存，并在反向传播中重用，这是高效实现链式法则的关键。

<a name='3'></a>
## 3 - 参数初始化 🎲

训练开始前，需要为网络的每一层初始化权重矩阵 $W$ 和偏置向量 $b$。

**初始化策略**：
- 权重 $W$ 用**小随机数**初始化（避免对称性问题）
- 偏置 $b$ 初始化为**零向量**

$$W^{[l]} \sim \mathcal{N}(0,1) \times 0.01, \quad b^{[l]} = \mathbf{0}$$

<a name='3-1'></a>
### 3.1 - 两层网络初始化

<a name='ex-1'></a>
### 练习 1 — `initialize_parameters`

实现一个**两层神经网络**的参数初始化函数。

网络结构：输入层（$n_x$ 个特征）→ 隐藏层（$n_h$ 个神经元）→ 输出层（$n_y$ 个输出）

需要初始化的参数：

| 参数 | 形状 | 初始化方式 |
|------|------|------------|
| $W_1$ | $(n_h, n_x)$ | 小随机数 × 0.01 |
| $b_1$ | $(n_h, 1)$ | 全零 |
| $W_2$ | $(n_y, n_h)$ | 小随机数 × 0.01 |
| $b_2$ | $(n_y, 1)$ | 全零 |

> 💡 提示：使用 `np.random.seed(1)` 确保结果可复现，使用 `np.random.randn` 生成随机矩阵。

In [ ]:
# GRADED FUNCTION: initialize_parameters

def initialize_parameters(n_x, n_h, n_y):
    """
    参数:
        n_x -- 输入层节点数
        n_h -- 隐藏层节点数
        n_y -- 输出层节点数

    返回:
        parameters -- 字典，包含 W1、b1、W2、b2
    """
    np.random.seed(1)

    ### START CODE HERE ###
    ### END CODE HERE ###

    parameters = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
    return parameters

In [ ]:
print("测试用例 1:\n")
parameters = initialize_parameters(3,2,1)
print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))
initialize_parameters_test_1(initialize_parameters)
print("\033[90m\n测试用例 2:\n")
parameters = initialize_parameters(4,3,2)
print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))
initialize_parameters_test_2(initialize_parameters)

**期望输出**
```
测试用例 1:

W1 = [[ 0.01624345 -0.00611756 -0.00528172]
 [-0.01072969  0.00865408 -0.02301539]]
b1 = [[0.]
 [0.]]
W2 = [[ 0.01744812 -0.00761207]]
b2 = [[0.]]
 All tests passed.

测试用例 2:

W1 = [[ 0.01624345 -0.00611756 -0.00528172 -0.01072969]
 [ 0.00865408 -0.02301539  0.01744812 -0.00761207]
 [ 0.00319039 -0.0024937   0.01462108 -0.02060141]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[-0.00322417 -0.00384054  0.01133769]
 [-0.01099891 -0.00172428 -0.00877858]]
b2 = [[0.]
 [0.]]
 All tests passed.
```

<a name='3-2'></a>
### 3.2 - L 层网络初始化

<a name='ex-2'></a>
### 练习 2 — `initialize_parameters_deep`

将初始化推广到任意层数 $L$ 的深度网络。

输入参数 `layer_dims` 是一个列表，例如 `[5, 4, 3]` 表示：
- 输入层：5 个特征
- 隐藏层：4 个神经元
- 输出层：3 个神经元

对第 $l$ 层（$l = 1, \ldots, L$）：

$$W^{[l]} \in \mathbb{R}^{\text{layer\_dims}[l] \times \text{layer\_dims}[l-1]}, \quad b^{[l]} \in \mathbb{R}^{\text{layer\_dims}[l] \times 1}$$

> 💡 提示：使用 `np.random.seed(3)` 保证可复现。循环中用字符串拼接构建键名，如 `'W' + str(l)`。

In [ ]:
# GRADED FUNCTION: initialize_parameters_deep

def initialize_parameters_deep(layer_dims):
    """
    参数:
        layer_dims -- 列表，每个元素为对应层的节点数

    返回:
        parameters -- 字典，包含 W1、b1、...、WL、bL
    """
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims)

    ### START CODE HERE ###
    ### END CODE HERE ###

    return parameters

In [ ]:
print("测试用例 1:\n")
parameters = initialize_parameters_deep([5,4,3])
print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))
initialize_parameters_deep_test_1(initialize_parameters_deep)
print("\033[90m\n测试用例 2:\n")
parameters = initialize_parameters_deep([4,3,2])
print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))
initialize_parameters_deep_test_2(initialize_parameters_deep)

**期望输出**
```
测试用例 1:

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493 -0.00277388]
 [-0.00354759 -0.00082741 -0.00627001 -0.00043818 -0.00477218]
 [-0.01313865  0.00884622  0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477  0.00982367 -0.01101068]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.01185047 -0.0020565   0.01486148  0.00236716]
 [-0.01023785 -0.00712993  0.00625245 -0.00160513]
 [-0.00768836 -0.00230031  0.00745056  0.01976111]]
b2 = [[0.]
 [0.]
 [0.]]
 All tests passed.

测试用例 2:

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493]
 [-0.00277388 -0.00354759 -0.00082741 -0.00627001]
 [-0.00043818 -0.00477218 -0.01313865  0.00884622]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[ 0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477]]
b2 = [[0.]
 [0.]]
 All tests passed.
```

<a name='4'></a>
## 4 - 前向传播模块 ➡️

前向传播将输入 $X$ 逐层变换，最终输出预测值 $\hat{Y} = A^{[L]}$。

我们将分三步实现：
1. **线性前向**：$Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$
2. **线性激活前向**：将线性变换与激活函数组合
3. **L 层模型前向**：堆叠所有层，完成完整的前向传播

<a name='4-1'></a>
### 4.1 - 线性前向

<a name='ex-3'></a>
### 练习 3 — `linear_forward`

实现单层的**线性变换**部分：

$$Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$$

其中：
- $A^{[l-1]}$：上一层的激活值（或输入 $X$），形状 $(n^{[l-1]}, m)$
- $W^{[l]}$：当前层权重，形状 $(n^{[l]}, n^{[l-1]})$
- $b^{[l]}$：当前层偏置，形状 $(n^{[l]}, 1)$
- $m$：样本数量

函数需同时返回 **cache** = $(A, W, b)$，供反向传播使用。

> 💡 提示：使用 `np.dot` 进行矩阵乘法。广播机制会自动处理偏置的维度对齐。

In [ ]:
# GRADED FUNCTION: linear_forward

def linear_forward(A, W, b):
    """
    实现一层的线性前向传播：Z = W·A + b

    参数:
        A -- 上一层激活值，形状 (上一层节点数, 样本数)
        W -- 权重矩阵，形状 (当前层节点数, 上一层节点数)
        b -- 偏置向量，形状 (当前层节点数, 1)

    返回:
        Z     -- 激活函数的输入（预激活值）
        cache -- (A, W, b)，供反向传播使用
    """
    ### START CODE HERE ###
    ### END CODE HERE ###

    cache = (A, W, b)
    return Z, cache

In [ ]:
t_A, t_W, t_b = linear_forward_test_case()
t_Z, t_linear_cache = linear_forward(t_A, t_W, t_b)
print("Z = " + str(t_Z))
linear_forward_test(linear_forward)

**期望输出**
```
Z = [[ 3.26295337 -1.23429987]]
```

<a name='4-2'></a>
### 4.2 - 线性激活前向

<a name='ex-4'></a>
### 练习 4 — `linear_activation_forward`

将线性变换与激活函数**组合**成一个模块：

$$A^{[l]} = g(Z^{[l]}) = g(W^{[l]} A^{[l-1]} + b^{[l]})$$

本实验支持两种激活函数，通过字符串参数 `activation` 切换：

| 激活函数 | 公式 | 适用场景 |
|----------|------|----------|
| `"sigmoid"` | $\sigma(Z) = \frac{1}{1+e^{-Z}}$ | 输出层（二分类） |
| `"relu"` | $\text{ReLU}(Z) = \max(0, Z)$ | 隐藏层 |

函数返回 **cache** = (linear_cache, activation_cache)，其中：
- `linear_cache` = $(A_{prev}, W, b)$
- `activation_cache` = $Z$（激活函数的反向传播需要它）

In [ ]:
# GRADED FUNCTION: linear_activation_forward

def linear_activation_forward(A_prev, W, b, activation):
    """
    实现 LINEAR -> ACTIVATION 的前向传播。

    参数:
        A_prev     -- 上一层激活值
        W          -- 权重矩阵
        b          -- 偏置向量
        activation -- 激活函数类型："sigmoid" 或 "relu"

    返回:
        A     -- 激活后的输出值
        cache -- (linear_cache, activation_cache)
    """
    if activation == "sigmoid":
        ### START CODE HERE ###
        ### END CODE HERE ###

    elif activation == "relu":
        ### START CODE HERE ###
        ### END CODE HERE ###

    cache = (linear_cache, activation_cache)
    return A, cache

In [ ]:
t_A_prev, t_W, t_b = linear_activation_forward_test_case()
t_A, t_linear_activation_cache = linear_activation_forward(t_A_prev, t_W, t_b, activation = "sigmoid")
print("With sigmoid: A = " + str(t_A))
t_A, t_linear_activation_cache = linear_activation_forward(t_A_prev, t_W, t_b, activation = "relu")
print("With ReLU: A = " + str(t_A))
linear_activation_forward_test(linear_activation_forward)

**期望输出**
```
With sigmoid: A = [[0.96890023 0.11013289]]
With ReLU: A = [[3.43896131 0.        ]]
```

<a name='4-3'></a>
### 4.3 - L 层模型前向

<a name='ex-5'></a>
### 练习 5 — `L_model_forward`

将所有层的前向传播串联起来，实现完整的 L 层前向传播：

$$[\text{LINEAR} \to \text{ReLU}]^{\times(L-1)} \to \text{LINEAR} \to \text{Sigmoid}$$

**实现步骤**：
1. 循环 $l = 1, \ldots, L-1$：调用 `linear_activation_forward` (relu)，将 cache 添加到 `caches` 列表
2. 对第 $L$ 层：调用 `linear_activation_forward` (sigmoid)，将 cache 添加到 `caches`
3. 返回最终输出 $A^{[L]}$ 和所有 caches

> 💡 提示：层数 $L = $ `len(parameters) // 2`，因为每层有 W 和 b 两个参数。
> 第 $l$ 层的参数键名为 `'W' + str(l)` 和 `'b' + str(l)`。

In [ ]:
# GRADED FUNCTION: L_model_forward

def L_model_forward(X, parameters):
    """
    实现 [LINEAR->ReLU]*(L-1)->LINEAR->Sigmoid 的完整前向传播。

    参数:
        X          -- 输入数据，形状 (输入特征数, 样本数)
        parameters -- initialize_parameters_deep() 的输出

    返回:
        AL     -- 最后一层的激活输出（预测概率）
        caches -- 所有层的 cache 列表，共 L 个
    """
    caches = []
    A = X
    L = len(parameters) // 2

    ### START CODE HERE ###
    # 前 L-1 层：LINEAR -> ReLU

    # 最后一层：LINEAR -> Sigmoid
    ### END CODE HERE ###

    return AL, caches

In [ ]:
t_X, t_parameters = L_model_forward_test_case_2hidden()
t_AL, t_caches = L_model_forward(t_X, t_parameters)
print("AL = " + str(t_AL))
print("Length of caches list = " + str(len(t_caches)))
L_model_forward_test(L_model_forward)

**期望输出**
```
AL = [[0.03921668 0.70498921 0.19734387 0.04728177]]
```

<a name='5'></a>
## 5 - 损失函数 📉

<a name='ex-6'></a>
### 练习 6 — `compute_cost`

使用**二元交叉熵损失**衡量预测与真实标签之间的差距：

$$\mathcal{L}(\hat{Y}, Y) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log \hat{y}^{(i)} + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

其中：
- $\hat{Y} = A^{[L]}$：模型输出的预测概率，形状 $(1, m)$
- $Y$：真实标签，形状 $(1, m)$
- $m$：样本数

> 💡 提示：使用 `np.sum` 和 `np.log`，最后用 `np.squeeze` 去除冗余维度。

In [ ]:
# GRADED FUNCTION: compute_cost

def compute_cost(AL, Y):
    """
    计算二元交叉熵损失。

    参数:
        AL -- 预测概率，形状 (1, 样本数)
        Y  -- 真实标签，形状 (1, 样本数)

    返回:
        cost -- 标量，交叉熵损失值
    """
    m = Y.shape[1]

    ### START CODE HERE ###
    ### END CODE HERE ###

    cost = np.squeeze(cost)
    return cost

In [ ]:
t_Y, t_AL = compute_cost_test_case()
t_cost = compute_cost(t_AL, t_Y)
print("Cost: " + str(t_cost))
compute_cost_test(compute_cost)

**期望输出**

| **cost** | 0.2797765635793422 |
|----------|--------------------|


<a name='6'></a>
## 6 - 反向传播模块 ⬅️

反向传播是神经网络学习的核心——它通过**链式法则**计算每个参数对损失的梯度。

与前向传播的模块一一对应：

| 前向 | 反向 |
|------|------|
| `linear_forward` | `linear_backward` |
| `linear_activation_forward` | `linear_activation_backward` |
| `L_model_forward` | `L_model_backward` |

在反向传播中，我们将从最后一层向第一层逐步计算梯度，并在每一步使用前向传播缓存的 cache。

<a name='6-1'></a>
### 6.1 - 线性反向

<a name='ex-7'></a>
### 练习 7 — `linear_backward`

已知当前层的梯度 $dZ^{[l]}$，利用链式法则计算：

$$dW^{[l]} = \frac{\partial \mathcal{L}}{\partial W^{[l]}} = \frac{1}{m} dZ^{[l]} (A^{[l-1]})^T$$

$$db^{[l]} = \frac{\partial \mathcal{L}}{\partial b^{[l]}} = \frac{1}{m} \sum_{i=1}^{m} dZ^{[l](i)}$$

$$dA^{[l-1]} = \frac{\partial \mathcal{L}}{\partial A^{[l-1]}} = (W^{[l]})^T dZ^{[l]}$$

其中 $m = A^{[l-1]}$.shape[1] 为样本数。

> 💡 提示：计算 $db$ 时记得使用 `np.sum(..., axis=1, keepdims=True)` 保持维度一致。

**`axis` 和 `keepdims` 的作用示意：**

In [ ]:
A = np.array([[1, 2], [3, 4]])
print('axis=1 and keepdims=True')
print(np.sum(A, axis=1, keepdims=True))
print('axis=1 and keepdims=False')
print(np.sum(A, axis=1, keepdims=False))

In [ ]:
# GRADED FUNCTION: linear_backward

def linear_backward(dZ, cache):
    """
    实现单层线性部分的反向传播。

    参数:
        dZ    -- 当前层输出 Z 的梯度
        cache -- 前向传播存储的 (A_prev, W, b)

    返回:
        dA_prev -- 上一层激活值的梯度
        dW      -- 当前层权重的梯度
        db      -- 当前层偏置的梯度
    """
    A_prev, W, b = cache
    m = A_prev.shape[1]

    ### START CODE HERE ###
    ### END CODE HERE ###

    return dA_prev, dW, db

In [ ]:
t_dZ, t_linear_cache = linear_backward_test_case()
t_dA_prev, t_dW, t_db = linear_backward(t_dZ, t_linear_cache)
print("dA_prev: " + str(t_dA_prev))
print("dW: " + str(t_dW))
print("db: " + str(t_db))
linear_backward_test(linear_backward)

**期望输出**
```
dA_prev: [[-1.15171336  0.06718465 -0.3204696   2.09812712]
 [ 0.60345879 -3.72508701  5.81700741 -3.84326836]
 [-0.4319552  -1.30987417  1.72354705  0.05070578]
 [-0.38981415  0.60811244 -1.25938424  1.47191593]
 [-2.52214926  2.67882552 -0.67947465  1.48119548]]
dW: [[ 0.07313866 -0.0976715  -0.87585828  0.73763362  0.00785716]
 [ 0.85508818  0.37530413 -0.59912655  0.71278189 -0.58931808]
 [ 0.97913304 -0.24376494 -0.08839671  0.55151192 -0.10290907]]
db: [[-0.14713786]
 [-0.11313155]
 [-0.13209101]]
```

<a name='6-2'></a>
### 6.2 - 线性激活反向

<a name='ex-8'></a>
### 练习 8 — `linear_activation_backward`

将激活函数的反向传播与线性层的反向传播组合：

$$dZ^{[l]} = dA^{[l]} \odot g'(Z^{[l]})$$

然后调用 `linear_backward(dZ, linear_cache)` 得到 $dA^{[l-1]}$、$dW^{[l]}$、$db^{[l]}$。

激活函数的反向传播由 `dnn_utils.py` 提供：
- `sigmoid_backward(dA, activation_cache)`
- `relu_backward(dA, activation_cache)`

In [ ]:
# GRADED FUNCTION: linear_activation_backward

def linear_activation_backward(dA, cache, activation):
    """
    实现 LINEAR->ACTIVATION 层的反向传播。

    参数:
        dA         -- 当前层激活值的梯度
        cache      -- (linear_cache, activation_cache)
        activation -- 激活函数类型："sigmoid" 或 "relu"

    返回:
        dA_prev -- 上一层激活值的梯度
        dW      -- 当前层权重的梯度
        db      -- 当前层偏置的梯度
    """
    linear_cache, activation_cache = cache

    if activation == "relu":
        ### START CODE HERE ###
        ### END CODE HERE ###

    elif activation == "sigmoid":
        ### START CODE HERE ###
        ### END CODE HERE ###

    return dA_prev, dW, db

In [ ]:
t_dAL, t_linear_activation_cache = linear_activation_backward_test_case()
t_dA_prev, t_dW, t_db = linear_activation_backward(t_dAL, t_linear_activation_cache, activation = "sigmoid")
print("With sigmoid: dA_prev = " + str(t_dA_prev))
print("With sigmoid: dW = " + str(t_dW))
print("With sigmoid: db = " + str(t_db))
t_dA_prev, t_dW, t_db = linear_activation_backward(t_dAL, t_linear_activation_cache, activation = "relu")
print("With relu: dA_prev = " + str(t_dA_prev))
print("With relu: dW = " + str(t_dW))
print("With relu: db = " + str(t_db))
linear_activation_backward_test(linear_activation_backward)

**期望输出**
```
With sigmoid: dA_prev = [[ 0.11017994  0.01105339]
 [ 0.09466817  0.00949723]
 [-0.05743092 -0.00576154]]
With sigmoid: dW = [[ 0.10266786  0.09778551 -0.01968084]]
With sigmoid: db = [[-0.05729622]]
With relu: dA_prev = [[ 0.44090989  0.        ]
 [ 0.37883606  0.        ]
 [-0.2298228   0.        ]]
With relu: dW = [[ 0.44513824  0.37371418 -0.10478989]]
With relu: db = [[-0.20837892]]
```

<a name='6-3'></a>
### 6.3 - L 层模型反向

<a name='ex-9'></a>
### 练习 9 — `L_model_backward`

实现完整的 L 层反向传播。

**第一步**：计算损失对最后一层输出 $A^{[L]}$ 的初始梯度：

$$dA^{[L]} = -\frac{Y}{A^{[L]}} + \frac{1-Y}{1-A^{[L]}}$$

**第二步**：对第 $L$ 层（Sigmoid），调用 `linear_activation_backward(..., 'sigmoid')`。

**第三步**：对 $l = L-2, \ldots, 0$（ReLU 层），逐层调用 `linear_activation_backward(..., 'relu')`。

所有梯度存储在字典 `grads` 中，键名格式为：`'dA' + str(l)`、`'dW' + str(l)`、`'db' + str(l)`。

In [ ]:
# GRADED FUNCTION: L_model_backward

def L_model_backward(AL, Y, caches):
    """
    实现 [LINEAR->ReLU]*(L-1)->LINEAR->Sigmoid 的完整反向传播。

    参数:
        AL     -- 前向传播的最终输出
        Y      -- 真实标签
        caches -- L_model_forward() 返回的 caches 列表

    返回:
        grads -- 梯度字典，包含 dA、dW、db
    """
    grads = {}
    L = len(caches)
    m = AL.shape[1]
    Y = Y.reshape(AL.shape)

    ### START CODE HERE ###
    # 初始化反向传播：计算损失对 AL 的梯度

    # 最后一层（Sigmoid）的梯度

    # 从 L-2 到 0 层（ReLU）逐层反向传播
    ### END CODE HERE ###

    return grads

In [ ]:
t_AL, t_Y_assess, t_caches = L_model_backward_test_case()
grads = L_model_backward(t_AL, t_Y_assess, t_caches)
print("dA0 = " + str(grads['dA0']))
print("dA1 = " + str(grads['dA1']))
print("dW1 = " + str(grads['dW1']))
print("dW2 = " + str(grads['dW2']))
print("db1 = " + str(grads['db1']))
print("db2 = " + str(grads['db2']))
L_model_backward_test(L_model_backward)

**期望输出**
```
dA0 = [[ 0.          0.52257901]
 [ 0.         -0.3269206 ]
 [ 0.         -0.32070404]
 [ 0.         -0.74079187]]
dA1 = [[ 0.12913162 -0.44014127]
 [-0.14175655  0.48317296]
 [ 0.01663708 -0.05670698]]
dW1 = [[0.41010002 0.07807203 0.13798444 0.10502167]
 [0.         0.         0.         0.        ]
 [0.05283652 0.01005865 0.01777766 0.0135308 ]]
dW2 = [[-0.39202432 -0.13325855 -0.04601089]]
db1 = [[-0.22007063]
 [ 0.        ]
 [-0.02835349]]
db2 = [[0.15187861]]
```

<a name='6-4'></a>
### 6.4 - 参数更新

<a name='ex-10'></a>
### 练习 10 — `update_parameters`

使用**梯度下降**更新所有参数：

$$W^{[l]} \leftarrow W^{[l]} - \alpha \cdot dW^{[l]}$$

$$b^{[l]} \leftarrow b^{[l]} - \alpha \cdot db^{[l]}$$

其中 $\alpha$ 为学习率（learning rate），控制每次更新的步长。

> ⚠️ 注意：为避免修改原始参数字典，使用 `copy.deepcopy(params)` 创建深拷贝后再更新。

In [ ]:
# GRADED FUNCTION: update_parameters

def update_parameters(params, grads, learning_rate):
    """
    使用梯度下降更新模型参数。

    参数:
        params        -- 当前参数字典（W1, b1, ..., WL, bL）
        grads         -- L_model_backward() 返回的梯度字典
        learning_rate -- 学习率 alpha

    返回:
        parameters -- 更新后的参数字典
    """
    parameters = copy.deepcopy(params)
    L = len(parameters) // 2

    ### START CODE HERE ###
    ### END CODE HERE ###

    return parameters

In [ ]:
t_parameters, grads = update_parameters_test_case()
t_parameters = update_parameters(t_parameters, grads, 0.1)
print("W1 = "+ str(t_parameters["W1"]))
print("b1 = "+ str(t_parameters["b1"]))
print("W2 = "+ str(t_parameters["W2"]))
print("b2 = "+ str(t_parameters["b2"]))
update_parameters_test(update_parameters)

**期望输出**
```
W1 = [[-0.59562069 -0.09991781 -2.14584584  1.82662008]
 [-1.76569676 -0.80627147  0.51115557 -1.18258802]
 [-1.0535704  -0.86128581  0.68284052  2.20374577]]
b1 = [[-0.04659241]
 [-1.28888275]
 [ 0.53405496]]
W2 = [[-0.55569196  0.0354055   1.32964895]]
b2 = [[-0.84610769]]
```

<a name='7'></a>
## 7 - 总结 🎉

恭喜你！你已经从零构建了一个完整的深度神经网络的所有核心模块：

| 模块 | 函数 |
|------|------|
| 参数初始化 | `initialize_parameters`、`initialize_parameters_deep` |
| 前向传播 | `linear_forward`、`linear_activation_forward`、`L_model_forward` |
| 损失计算 | `compute_cost` |
| 反向传播 | `linear_backward`、`linear_activation_backward`、`L_model_backward` |
| 参数更新 | `update_parameters` |

这些模块就像乐高积木——你可以把它们拼在一起，搭建任意层数的神经网络来解决实际问题（如图像分类、文本识别等）。

**下一步**：将这些模块组合成完整的训练循环，在真实数据集上训练你的第一个深度网络！ 🚀